

To install environment, `conda env create -f environment-HGQ.yml`, or to update env `conda env update -f environment-HGQ.yml`. Remember to restart kernel.

In [16]:
import os
model_to_test = 'MNIST_hgq2'
model_revision = 'Training_AdaptiveHP'
hls4ml_revision = 'VU_fifo-opt'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, model_revision)
os.makedirs(model_dir, exist_ok=True)

description = f"""
# Model Configuration


- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [17]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import os
from sklearn.metrics import accuracy_score
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']


In [18]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_test = X_test.astype("float32") / 255
y_test = keras.utils.to_categorical(y_test, 10)
y_test.dtype

dtype('float64')

In [19]:
# Prepare subset of testdata for simulation (running everything takes an unnecessary long time)
simulation_rows = 100
x_test_sim_path = os.path.join(base_dir, "x_test_sim.npy")
y_test_sim_path = os.path.join(base_dir, "y_test_sim.npy")
np.save(x_test_sim_path, X_test[:simulation_rows])
np.save(y_test_sim_path, y_test[:simulation_rows])

Load existing model

In [20]:
from keras.models import load_model
#from qkeras.utils import load_qmodel
import hgq.layers

#keras_model_path = os.path.join(base_dir, "/home/ncgadmin/DAT255/DAT255-project/MNIST/CNN_HGQ_StaticTraining/test_models/epoch=2619-val_acc=0.974-ebops=48591-val_loss=0.091.keras")
#keras_model_path = os.path.join(base_dir, "/home/ncgadmin/DAT255/DAT255-project/MNIST/CNN_HGQ_StaticTraining/test_models/epoch=3277-val_acc=0.967-ebops=30992-val_loss=0.120.keras")
#keras_model_path = os.path.join(base_dir, "/home/ncgadmin/DAT255/DAT255-project/SVHN/Qkeras/Model_Qkeras_NoBatchNorm.h5")
keras_model_path = "MNIST_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.9578_ebops=34304.keras"
#model = load_qmodel(keras_model_path)
model = load_model(keras_model_path)
score = model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9578 - loss: 0.1440


In [21]:
from hgq.utils import trace_minmax

trace_minmax(model, X_test, verbose=True)

conv0 : 5408
conv1 : 21004
dense0: 7892
Total: 34304


34304

In [22]:
# Save the model summary to a text file
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv0 (QConv2D)                 │ (None, 26, 26, 16)     │           646 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool0 (MaxPooling2D)            │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (QConv2D)                 │ (None, 11, 11, 16)     │         9,286 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense0 (QDense)                 │ (None, 10)             │        16,046 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 64,926 (234.60 KB)

 Trainable params: 19,473 (76.07 KB)

 Non-trainable params: 6,505 (6.39 KB)

 Optimizer params: 38,948 (152.14 KB)

In [ ]:
import keras

inputs = keras.Input(shape=(32, 32, 3), name="hw_input")

x = model.get_layer('qconv0')(inputs)
x = model.get_layer('relu0')(x)
x = model.get_layer('pool0')(x)

x = model.get_layer('qconv1')(x)
x = model.get_layer('relu1')(x)
x = model.get_layer('pool1')(x)

x = model.get_layer('qconv2')(x)
x = model.get_layer('relu2')(x)
x = model.get_layer('pool2')(x)

x = keras.layers.Flatten()(x)

x = model.get_layer('qdense0')(x)
x = model.get_layer('relu3')(x)

x = model.get_layer('dropout')(x) 

x = model.get_layer('qdense1')(x)
outputs = model.get_layer('softmax')(x)


model_stripped = keras.Model(inputs=inputs, outputs=outputs)

model_stripped.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hw_input (InputLayer)       [(None, 32, 32, 3)]       0         
                                                                 
 qconv0 (QConv2D)            (None, 30, 30, 32)        896       
                                                                 
 relu0 (QActivation)         (None, 30, 30, 32)        0         
                                                                 
 pool0 (MaxPooling2D)        (None, 15, 15, 32)        0         
                                                                 
 qconv1 (QConv2D)            (None, 13, 13, 64)        18496     
                                                                 
 relu1 (QActivation)         (None, 13, 13, 64)        0         
                                                                 
 pool1 (MaxPooling2D)        (None, 6, 6, 64)          0     

# Convert and synthesize with HLS4ML
Configure parameters.
KV260: xck26-sfvc784-2LV-c 

In [23]:
import hls4ml
#import plotting

config = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='vitisunified')
config['Model']['ReuseFactor'] = 1
#config['Model']['Strategy'] = 'Resource'
#config['Model']['Strategy'] = 'Distributed Arithmetic'

proj_name = f"{str(model_to_test)}_{str(model_revision)}_hls4ml_prj_{str(hls4ml_revision)}"

print("-----------------------------------")
print("Configuration")
#plotting.print_dict(config)
print("-----------------------------------")

hls_model = hls4ml.converters.convert_from_keras_model(
    model,    
    backend='vitisunified',
    hls_config=config,
    io_type='io_stream',
    proj_name = proj_name,
    output_dir=output_dir, 
    board       = 'kv260',
    part='xck26-sfvc784-2LV-c',
    clock_period='5',
)

hls_model.compile()
#hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True,to_file=os.path.join(output_dir, "model-plot.png"))

-----------------------------------
Configuration
-----------------------------------


Check performance

In [24]:
y_keras = model.predict(X_test)
y_hls = hls_model.predict(np.ascontiguousarray(X_test))

print("Difference in inference-calculations between Keras-model and HLS4ML-compiled model (first rows):")
for x,y in enumerate(y_keras[:5]):
    print(f"{y-y_hls[x]}")

print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))



313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Difference in inference-calculations between Keras-model and HLS4ML-compiled model (first rows):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Keras  Accuracy: 0.9578
hls4ml Accuracy: 0.9578


In [25]:
max_diff = np.max(np.abs(y_keras - y_hls))
print("Max Bit-Error: {:.6f}".format(max_diff))

Max Bit-Error: 0.000000


In [28]:
hls_model.build(
    csim=False,
    synth=True, 
    reset=True,
    fifo_opt=True,
    bitfile=True,
    vsynth=True,
    ) 


****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Mon May 18 15:25:25 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/MNIST_CNN/MNIST_hgq2/Training_AdaptiveHP/hls4ml_prj_VU_fifo-opt/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/MNIST_CNN/MNIST_hgq2/Training_AdaptiveHP/hls4ml_prj_VU_fifo-opt/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/MNIST_CNN/MNIST_hgq2/Training_AdaptiveHP/hls4ml_prj_VU_fifo-opt/vitis_workspace/myproject/vitis_unified_project'.
INFO: [HLS 200-1505] Using

In [28]:
hls4ml.report.read_vivado_report(os.path.join(output_dir))

Unable to read project data. Exiting.
